# Auditoria de imagenes nuevas contra mosaic dataset

Notebook de control para comparar una carpeta raiz de entrada contra un mosaic dataset, usando una subcarpeta de confianza indicada por el cliente.

In [ ]:
from datetime import datetime
from pathlib import Path

import pandas as pd

from core.mosaic_image_audit import *

# PARAMETROS PRINCIPALES
PATH_INPUT_RAIZ = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_Drone_Sin_Procesar\INPUT"
PATH_MOSAIC_DATASET = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.CL_MLP_PAO_IF_Ortho_Geosupport"

# Subfolder de confianza informado por el cliente dentro de PATH_INPUT_RAIZ.
SUBFOLDER_CONTROL_CLIENTE = "20260206_Geosupport_primera entrega"

# Usar None para leer toda la tabla exportada. Puede bajarse para pruebas rapidas.
MAX_EXPORTED_PATH_ROWS = None

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_results_dir = Path.cwd() / "outputs" / "auditoria_mosaico" / run_timestamp

print("Input raiz:", PATH_INPUT_RAIZ)
print("Subfolder control cliente:", SUBFOLDER_CONTROL_CLIENTE)
print("Mosaic dataset:", PATH_MOSAIC_DATASET)
print("Salida:", output_results_dir)

## 1. Buscar imagenes en el input raiz

La busqueda es recursiva. El flag `is_in_control_folder` identifica las imagenes dentro de la entrega que el cliente marco como confiable.

In [ ]:
input_images_df = scan_input_images(PATH_INPUT_RAIZ)
input_images_df = add_control_flags(input_images_df, SUBFOLDER_CONTROL_CLIENTE)

ortho_input_images_df = input_images_df[input_images_df["extension"].isin(ORTHO_MOSAIC_EXTENSIONS)].copy()
control_ortho_images_df = ortho_input_images_df[ortho_input_images_df["is_in_control_folder"]].copy()

print(f"Imagenes encontradas en input raiz: {len(input_images_df)}")
print(f"TIF/TIFF en input raiz: {len(ortho_input_images_df)}")
print(f"TIF/TIFF en subfolder de control cliente: {len(control_ortho_images_df)}")

display(input_images_df.groupby(["extension", "is_in_control_folder"]).size().reset_index(name="count"))
display(ortho_input_images_df.groupby(["top_folder", "is_in_control_folder"]).size().reset_index(name="count"))

## 2. Exportar paths oficiales del mosaic dataset

Se usa `arcpy.management.ExportMosaicDatasetPaths` para obtener los nombres/rutas reales que quedaron cargados en el mosaico. Esta tabla es la fuente de control para comparar contra el nombre esperado que genera el script.

In [ ]:
mosaic_paths_df, mosaic_paths_fields_df, exported_mosaic_paths_table = export_mosaic_dataset_paths_to_dataframe(
    PATH_MOSAIC_DATASET,
    max_rows=MAX_EXPORTED_PATH_ROWS,
)
candidate_path_fields = detect_candidate_path_fields(mosaic_paths_fields_df)
mosaic_image_inventory_df = build_mosaic_image_inventory(mosaic_paths_df, candidate_path_fields)

print(f"Tabla exportada por ExportMosaicDatasetPaths: {exported_mosaic_paths_table}")
print(f"Registros exportados de paths del mosaic dataset: {len(mosaic_paths_df)}")
print(f"Campos candidatos de path/name en tabla exportada: {candidate_path_fields}")
print(f"Entradas normalizadas del inventario: {len(mosaic_image_inventory_df)}")

display(mosaic_paths_fields_df)
display(mosaic_paths_df.head(25))
display(mosaic_image_inventory_df.head(25))

## 3. Comparar nombres esperados contra el mosaico

`load_status = nueva_candidata` significa que el nombre esperado por el script no aparece en el mosaico. El triage separa las candidatas donde la fecha ya existe en el mosaico, porque ahi puede haber un cambio de nomenclatura en la carga manual.

In [ ]:
input_expected_names_df = add_expected_names(ortho_input_images_df)
input_vs_mosaic_df = add_mosaic_match(input_expected_names_df, mosaic_image_inventory_df)
triage_df = add_triage(input_vs_mosaic_df, mosaic_image_inventory_df)

script_new_df = triage_df[triage_df["load_status"] == "nueva_candidata"].copy()
script_new_control_df = script_new_df[script_new_df["is_in_control_folder"]].copy()
high_confidence_new_control_df = script_new_control_df[script_new_control_df["triage_status"] == "nueva_alta_confianza"].copy()
same_date_review_control_df = script_new_control_df[script_new_control_df["triage_status"] == "revisar_fecha_existente_en_mosaico"].copy()
review_or_discard_control_df = triage_df[
    triage_df["is_in_control_folder"]
    & triage_df["load_status"].isin(["sin_fecha", "sin_sector", "descartar_posible_plano"])
].copy()

print("Estados globales:")
display(triage_df["load_status"].value_counts(dropna=False).reset_index(name="count").rename(columns={"index": "load_status"}))

print("Triage dentro del subfolder de control:")
display(triage_df[triage_df["is_in_control_folder"]]["triage_status"].value_counts(dropna=False).reset_index(name="count").rename(columns={"index": "triage_status"}))

## 4. Resumen ejecutivo y muestras para analisis

La tabla de resumen responde: cuantas imagenes dice el script que son nuevas y cuantas caen dentro del directorio de control del cliente. Las muestras ayudan a ver si las que no aparecen en el mosaico fueron cargadas con otro nombre.

In [ ]:
summary_rows = [
    {"metric": "run_timestamp", "value": run_timestamp},
    {"metric": "input_root", "value": PATH_INPUT_RAIZ},
    {"metric": "client_control_subfolder", "value": SUBFOLDER_CONTROL_CLIENTE},
    {"metric": "mosaic_dataset", "value": PATH_MOSAIC_DATASET},
    {"metric": "input_images_count", "value": len(input_images_df)},
    {"metric": "root_ortho_tif_count", "value": len(ortho_input_images_df)},
    {"metric": "control_ortho_tif_count", "value": len(control_ortho_images_df)},
    {"metric": "exported_mosaic_paths_table", "value": exported_mosaic_paths_table},
    {"metric": "mosaic_exported_paths_count", "value": len(mosaic_paths_df)},
    {"metric": "mosaic_inventory_count", "value": len(mosaic_image_inventory_df)},
    {"metric": "script_new_count_all_root", "value": len(script_new_df)},
    {"metric": "script_new_count_inside_control", "value": len(script_new_control_df)},
    {"metric": "high_confidence_new_inside_control", "value": len(high_confidence_new_control_df)},
    {"metric": "same_date_review_inside_control", "value": len(same_date_review_control_df)},
    {"metric": "review_or_discard_inside_control", "value": len(review_or_discard_control_df)},
]

for status, count in triage_df["load_status"].value_counts(dropna=False).items():
    summary_rows.append({"metric": f"load_status_{status}", "value": int(count)})

for status, count in triage_df[triage_df["is_in_control_folder"]]["triage_status"].value_counts(dropna=False).items():
    summary_rows.append({"metric": f"control_triage_{status}", "value": int(count)})

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

sample_columns = [
    "triage_status", "load_status", "file_name", "relative_path", "expected_name",
    "expected_date_token", "expected_sector", "date_exists_in_mosaic",
    "same_date_mosaic_examples", "matched_mosaic_path"
]

print("Muestra: nuevas de alta confianza dentro del control")
display(high_confidence_new_control_df[sample_columns].head(30))

print("Muestra: dice nueva, pero la fecha ya existe en el mosaico")
display(same_date_review_control_df[sample_columns].head(30))

print("Muestra: revision o descarte dentro del control")
display(review_or_discard_control_df[sample_columns].head(30))

## 5. Exportar resultados

Se exportan CSV y SQLite para analizarlos fuera del servidor.

In [ ]:
dataframes_to_export = {
    "summary": summary_df,
    "input_images": input_images_df,
    "ortho_input_images": ortho_input_images_df,
    "control_ortho_images": control_ortho_images_df,
    "mosaic_paths_fields": mosaic_paths_fields_df,
    "mosaic_paths_export": mosaic_paths_df,
    "mosaic_image_inventory": mosaic_image_inventory_df,
    "input_vs_mosaic": input_vs_mosaic_df,
    "triage": triage_df,
    "script_new_all_root": script_new_df,
    "script_new_inside_control": script_new_control_df,
    "high_confidence_new_inside_control": high_confidence_new_control_df,
    "same_date_review_inside_control": same_date_review_control_df,
    "review_or_discard_inside_control": review_or_discard_control_df,
}

exported_results = export_results(dataframes_to_export, output_results_dir, run_timestamp)
exported_results_df = pd.DataFrame([{"name": name, "path": str(path)} for name, path in exported_results.items()])

print(f"Resultados exportados en: {output_results_dir}")
display(exported_results_df)